<a href="https://colab.research.google.com/github/reems256/AI_intrusion_detection/blob/main/AI_project_final_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_selection import RFE
from sklearn import tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

In [ ]:
from google.colab import files
uploaded = files.upload()
df = pd.read_csv("intrusion.csv")

Saving intrusion.csv to intrusion.csv


In [ ]:
# 1- Dataset preprocessing:
df['dur'] = pd.to_numeric(df['dur'], errors='coerce') #convert dur to float
df.drop_duplicates(inplace=True) #drops duplicates
df.dropna(inplace = True) #drop N/A

df = df.replace('-', pd.NA)
df.dropna(inplace=True) #drop rows with (-) as a value

df.drop(columns=['id', 'label'], inplace=True) #drop unnesseccary features
# Display category counts as a table
category_counts = df['attack_cat'].value_counts().reset_index()
category_counts.columns = ['Category', 'Number of Records']

print(category_counts)
print(df.shape)
print(df.isnull().sum())
print(df.describe())

         Category  Number of Records
0         Generic              57956
1          Normal              29113
2        Exploits              21480
3             DoS               2508
4         Fuzzers               2266
5  Reconnaissance               2207
6        Analysis                564
7           Worms                148
8        Backdoor                110
(116352, 43)
dur                  0
proto                0
service              0
state                0
spkts                0
dpkts                0
sbytes               0
dbytes               0
rate                 0
sttl                 0
dttl                 0
sload                0
dload                0
sloss                0
dloss                0
sinpkt               0
dinpkt               0
sjit                 0
djit                 0
swin                 0
stcpb                0
dtcpb                0
dwin                 0
tcprtt               0
synack               0
ackdat               0
smean              

In [ ]:
#encode the dataset
le = LabelEncoder()
df["proto"] = le.fit_transform(df["proto"])
df["service"] = le.fit_transform(df["service"])
df["state"] = le.fit_transform(df["state"])
df["attack_cat"] = le.fit_transform(df["attack_cat"])

In [ ]:
# 2 - Feature Selection:
x = df.iloc[: , 0:-1]
y = df.iloc[:, -1]
clf = tree.DecisionTreeClassifier()
rfe = RFE(clf , n_features_to_select= 10)
fit = rfe.fit(x,y)

x = x.iloc[:, fit.support_]
print(x)

# dataset splitting:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size = 0.3 , random_state = 42)

        sbytes  dbytes  sttl     dinpkt         sjit       stcpb       dtcpb  \
3          628     770    62  90.235726   259.080172  1107119177  1047442890   
11       56329    2212    62  75.092445  3253.278833  1824722662   860716719   
15         138       0   254   0.000000     0.000000           0           0   
17         860    1096    62  47.669145  2124.837873  3882971404  3084071099   
21         998     268   254  56.579801  1928.550710  2665974075  3521361798   
...        ...     ...   ...        ...          ...         ...         ...   
256803   68199     612   254  56.755152  1376.971154  3137145926  3197614932   
256807   68199     612   254  57.166383  1389.032442  3567303131  4168480199   
256859   68199     612   254  63.407922  1526.290325  1516200001  1237225352   
256881   68199     612   254  53.710461  1300.964841   860363568   341738851   
257535   68109    1026   254  96.738422  3675.096013  1739940755   196470105   

          ackdat  smean  ct_srv_dst  
3

In [ ]:
# 3 - Model design:
# 3.1: Decision Tree

DT_model=tree.DecisionTreeClassifier(random_state=42)
DT_model.fit(x_train,y_train)
DT_predictions= DT_model.predict(x_test)

print('The accuracy is: ',accuracy_score(y_test, DT_predictions))

print('The precision is:', precision_score(y_test, DT_predictions, average='weighted'))
print('The recall is:', recall_score(y_test, DT_predictions, average='weighted'))
print('The F1 score is:', f1_score(y_test, DT_predictions, average='weighted'))

The accuracy is:  0.9269753051051395
The precision is: 0.9281604027527606
The recall is: 0.9269753051051395
The F1 score is: 0.9275267747927762


In [ ]:
# 3.2: Random Forest
rf_model = RandomForestClassifier(n_estimators=111, max_depth=7, random_state=42)

rf_model.fit(x_train,y_train)
rf_predictions = rf_model.predict(x_test)

print("The accuracy is:", accuracy_score(y_test, rf_predictions))
print("The precision is:", precision_score(y_test, rf_predictions,average='weighted'))
print("The recall is:",recall_score(y_test, rf_predictions,average='weighted'))
print("The f1 score  is:", f1_score(y_test, rf_predictions,average='weighted'))

The accuracy is: 0.9260585572680915
The precision is: 0.9242321968162638
The recall is: 0.9260585572680915
The f1 score  is: 0.9085912055934039


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# 3.3 XGBoost:
XGB_model = XGBClassifier(objective='multi:softmax', num_class=9, tree_method='hist', eval_metric='mlogloss', n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample = 0.8,
    random_state=42,
)
XGB_model.fit(x_train, y_train)
XGB_predictions = XGB_model.predict(x_test)

print("The accuracy is:", accuracy_score(y_test, XGB_predictions))
print("The precision is:", precision_score(y_test, XGB_predictions,average='weighted'))
print("The recall is:",recall_score(y_test, XGB_predictions,average='weighted'))
print("The f1 score  is:", f1_score(y_test, XGB_predictions,average='weighted'))

The accuracy is: 0.9439924368303444
The precision is: 0.9405769766378842
The recall is: 0.9439924368303444
The f1 score  is: 0.9397860342467609


In [ ]:
# 3.4: MLP
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


model = MLPClassifier(solver="adam" , alpha= 0.001 , hidden_layer_sizes= (20) , activation="relu" , max_iter=500 , tol = 0.0001)
model.fit(x_train , y_train)

MLP_predictions = model.predict(x_test)

print("The accuracy is:", accuracy_score(y_test, MLP_predictions))
print("The precision is:", precision_score(y_test, MLP_predictions,average='weighted'))
print("The recall is:",recall_score(y_test, MLP_predictions,average='weighted'))
print("The f1 score  is:", f1_score(y_test, MLP_predictions,average='weighted'))

The accuracy is: 0.9258293703088294
The precision is: 0.9171578349072829
The recall is: 0.9258293703088294
The f1 score  is: 0.9160902138817935


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
